# Bank Statement Automation — Live Demo

**Gmail → Drive → Parse → Analyse → Visualise → Cross-check → Excel**

Click **Runtime → Run all** to see the full pipeline execute on synthetic bank statements.

---

In [ ]:
# Setup: clone repo, install deps, configure paths
import os, sys
if not os.path.exists('/content/bank-statement-automation'):
    !git clone https://github.com/RohitS2497/bank-statement-automation.git /content/bank-statement-automation
os.chdir('/content/bank-statement-automation')
sys.path.insert(0, '/content/bank-statement-automation')
!pip -q install pdfplumber pypdf reportlab openpyxl python-dateutil
print("✅ Setup complete. All files at: /content/bank-statement-automation/")

In [ ]:
# Stage 0: Generate synthetic bank statement PDFs (simulates what Gmail+Drive delivers)
!python /content/bank-statement-automation/gen_data.py
print("\n📂 Input PDFs generated (these simulate what the Apps Script fetches from Gmail):")
for f in sorted(os.listdir('inbox')):
    size = os.path.getsize(f'inbox/{f}')
    print(f"   📄 {f}  ({size:,} bytes)")

# Show input files (these simulate what the Apps Script drops into Drive)
print("\n📂 INPUT PDFs (inbox/):")
for f in sorted(os.listdir('inbox')):
    size = os.path.getsize(f'inbox/{f}')
    print(f"   {f}  ({size:,} bytes)")

In [ ]:
# Run the full 6-stage pipeline (Fetch → Parse → Analyse → Visualise → Cross-check → Export)
import importlib, statement_pipeline as sp
importlib.reload(sp)
sp.run()
print("\n✅ Pipeline complete. Scroll down to see results.")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 📊 RESULTS: Charts
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from IPython.display import Image, display, HTML
import os

display(HTML("<h2>📊 Visualisations</h2>"))
for chart in sorted(os.listdir('outputs/charts')):
    display(Image(filename=f'outputs/charts/{chart}', width=620))
    print()

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 🔍 CROSS-CHECK: Validation Flags (anomalies the pipeline caught)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import pandas as pd
from IPython.display import HTML

display(HTML("<h2>🔍 Validation Flags — Anomalies Detected</h2>"))
display(HTML("<p><em>The pipeline deliberately includes a corrupted balance and a duplicate "
             "transaction in the ICICI statement to demonstrate cross-check capability.</em></p>"))
flags = pd.read_excel('outputs/Bank_Statement_Analysis.xlsx', sheet_name='Validation Flags')
flags.style.set_properties(**{'text-align': 'left'}).set_table_styles(
    [{'selector': 'th', 'props': [('background-color', '#0C1C2C'), ('color', 'white')]}])

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 🛡️ EDGE CASES: File Status (how the pipeline handles bad inputs)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
display(HTML("<h2>🛡️ Edge Case Handling</h2>"))
display(HTML("<p><em>The pipeline doesn't crash on bad inputs — it classifies them and moves on.</em></p>"))
fs = pd.read_excel('outputs/Bank_Statement_Analysis.xlsx', sheet_name='File Status')
fs.style.set_properties(**{'text-align': 'left'}).set_table_styles(
    [{'selector': 'th', 'props': [('background-color', '#0C1C2C'), ('color', 'white')]}])

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 📋 SUMMARY: Transaction Overview + Category Breakdown
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
txns = pd.read_excel('outputs/Bank_Statement_Analysis.xlsx', sheet_name='Transactions')
cat = pd.read_excel('outputs/Bank_Statement_Analysis.xlsx', sheet_name='Category Summary')
monthly = pd.read_excel('outputs/Bank_Statement_Analysis.xlsx', sheet_name='Monthly Summary')

display(HTML(f"""<h2>📋 Transaction Summary</h2>
<table style="font-size:14px; border-collapse:collapse;">
<tr><td style="padding:4px 12px;"><b>Total transactions parsed</b></td><td>{len(txns)}</td></tr>
<tr><td style="padding:4px 12px;"><b>Total income</b></td><td>INR {txns['Credit'].sum():,.2f}</td></tr>
<tr><td style="padding:4px 12px;"><b>Total expense</b></td><td>INR {txns['Debit'].sum():,.2f}</td></tr>
<tr><td style="padding:4px 12px;"><b>Net</b></td><td>INR {txns['Credit'].sum() - txns['Debit'].sum():,.2f}</td></tr>
<tr><td style="padding:4px 12px;"><b>Statements processed</b></td><td>2 (HDFC + ICICI)</td></tr>
<tr><td style="padding:4px 12px;"><b>Edge cases handled</b></td><td>2 (encrypted PDF + blank scan)</td></tr>
</table>"""))

display(HTML("<h3>Category Breakdown</h3>"))
cat.style.set_properties(**{'text-align': 'left'}).set_table_styles(
    [{'selector': 'th', 'props': [('background-color', '#0C1C2C'), ('color', 'white')]}])

---

## How to inspect the files

After running, use the **file browser** (folder icon on the left sidebar) to navigate:
- **Input PDFs:** `/content/bank-statement-automation/inbox/`
- **Output Excel:** `/content/bank-statement-automation/outputs/Bank_Statement_Analysis.xlsx`
- **Charts:** `/content/bank-statement-automation/outputs/charts/`

Or use the download links in the cell below.

---

## Architecture

| Stage | Tool | Runs where |
|-------|------|------------|
| **Fetch + Store** | `FetchStatements.gs` (Apps Script) | Inside Google account, on hourly trigger |
| **Parse → Analyse → Visualise → Cross-check → Export** | `statement_pipeline.py` | Google Colab (this notebook) or local |

Apps Script has native Gmail+Drive access (no OAuth or Cloud project needed). Python handles parsing and analysis. The whole system stays inside your Google account.

### To connect to a real Gmail account:
1. Paste `FetchStatements.gs` into script.google.com → run once → approve permissions
2. Mount Drive in this notebook: `from google.colab import drive; drive.mount('/content/drive')`
3. Point pipeline at Drive folder: `sp.INBOX = '/content/drive/MyDrive/Bank Statements'`

Everything works unchanged on real data.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 📥 DOWNLOAD or SAVE TO DRIVE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from google.colab import files
from IPython.display import HTML

display(HTML("""<h2>📥 Download Output Files</h2>
<p>Click the buttons below to download the Excel and CSV to your machine.</p>"""))

print("Downloading Excel workbook...")
files.download('outputs/Bank_Statement_Analysis.xlsx')

print("Downloading clean CSV...")
files.download('outputs/transactions_clean.csv')

print("\n💡 Tip: You can also browse all files using the folder icon (📁) in the left sidebar.")
print("   Input PDFs:  /content/bank-statement-automation/inbox/")
print("   All outputs: /content/bank-statement-automation/outputs/")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ☁️ SAVE OUTPUT TO GOOGLE DRIVE (proves the end-to-end loop)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ⚠️ RUN THIS CELL MANUALLY (don't use Run All for this one)
# When prompted, click "Connect to Google Drive" and allow access.

from IPython.display import HTML
import shutil

display(HTML("""<h2>☁️ Save to Google Drive</h2>
<p><b>Instructions:</b> Run this cell → click "Connect to Google Drive" when prompted →
click "Allow". The output Excel will be saved directly to your Drive.</p>
<p>This completes the full loop: <b>Gmail → Drive → Parse → Excel back into Drive</b>.</p>"""))

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

OUTPUT_DRIVE_FOLDER = '/content/drive/MyDrive/Bank Statement Reports'
os.makedirs(OUTPUT_DRIVE_FOLDER, exist_ok=True)

shutil.copy('outputs/Bank_Statement_Analysis.xlsx', OUTPUT_DRIVE_FOLDER)
shutil.copy('outputs/transactions_clean.csv', OUTPUT_DRIVE_FOLDER)
shutil.copytree('outputs/charts', f'{OUTPUT_DRIVE_FOLDER}/charts', dirs_exist_ok=True)

print(f"\n✅ Output saved to Google Drive:")
print(f"   📁 Google Drive → Bank Statement Reports/")
print(f"      📊 Bank_Statement_Analysis.xlsx")
print(f"      📄 transactions_clean.csv")
print(f"      📈 charts/ (3 PNG files)")
print(f"\n🎯 Open drive.google.com → look for 'Bank Statement Reports' folder.")
print(f"   This is exactly what happens in production — the Excel lands in Drive automatically.")

---

## What this automation does (the simple version)

**Every hour, automatically:**

1. Apps Script scans Gmail for new bank statement emails
2. Saves PDF attachments into an organised Drive folder (`Bank Statements/2026/2026-06/`)
3. Python pipeline picks up the PDFs, parses them, and produces a structured Excel workbook
4. Excel lands back in Google Drive — ready to open

**Setup time:** 5 minutes (paste one script, run once, done).

**After setup:** Fully hands-off. New statements arrive → processed → Excel updated → saved to Drive.

---

## FAQ

**"Why two tools (Apps Script + Python)?"**
Apps Script is the only way to access Gmail and Drive without setting up OAuth credentials or a Cloud project. Python (via Colab) is far better at PDF parsing and data analysis. This split gives you the simplest setup with the strongest output.

**"Can this run fully automatically without me opening Colab?"**
Yes. In production, the Python pipeline runs as a scheduled Colab notebook (or a Cloud Function). The Apps Script already runs on its own hourly trigger. Both are fire-and-forget after initial setup.

**"What if a PDF is corrupted, password-protected, or not a real statement?"**
The pipeline handles it gracefully — classifies the file, logs the reason, and continues processing the rest. No crashes. See the File Status table above.

**"What if two banks use different column names?"**
Built-in header mapping. The parser recognises common variations (Narration/Particulars/Description, Withdrawal/Debit/Dr, etc.) and normalises them automatically.